# Grammy Awards Web Scraping

This notebook extracts Grammy Award winners from the official Grammy website using Python.

In [2]:
%pip install lxml

Note: you may need to restart the kernel to use updated packages.


In [3]:
import requests
from bs4 import BeautifulSoup
import pandas as pd


## 1. Configure the Target URL

Define the target page and the HTTP headers used to send the request.

In [4]:
url = "https://www.grammy.com/awards/62nd-annual-grammy-awards-2019/"

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept-Language": "en-US,en;q=0.9",
    "Accept": (
        "text/html,application/xhtml+xml,application/xml;q=0.9,"
        "image/webp,*/*;q=0.8"
    )
}

print(f"Connecting to: {url}")

Connecting to: https://www.grammy.com/awards/62nd-annual-grammy-awards-2019/


## 2. Download the HTML

Send an HTTP GET request to retrieve the page HTML. Check the HTTP response status before processing the content.

In [5]:
try:
    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    response.raise_for_status()

    print("Connection successful.")
    print(f"HTTP status code: {response.status_code}")
    print(f"HTML size: {len(response.content):,} bytes")

except requests.exceptions.RequestException as error:
    print(f"Connection error: {error}")
    raise

Connection successful.
HTTP status code: 200
HTML size: 342,706 bytes


## 3. Parse the HTML

Use BeautifulSoup to parse the HTML and make the page structure easier to inspect.

In [6]:
soup = BeautifulSoup(response.content, "html.parser")

page_title = (
    soup.title.get_text(strip=True)
    if soup.title
    else "Title not found"
)

print(f"Page title: {page_title}")

Page title: 62nd Annual Grammy Awards 2020 | Grammy


## 4. Find the Award Categories

Locate the main accordion container and identify the elements that represent award categories.

In [7]:
# Inspect HTML tables
tables = soup.find_all("table")

print(f"HTML tables found: {len(tables)}")

for index, table in enumerate(tables):
    first_row = table.find("tr")

    if first_row:
        headers = [
            cell.get_text(" ", strip=True)
            for cell in first_row.find_all(["th", "td"], recursive=False)
        ]

        print(f"Table {index}: {headers}")

HTML tables found: 1
Table 0: ['Category', 'Winner', 'Title', 'Actions']


In [8]:
main_container = soup.find(
    "div",
    id="cer-acc-38187"
)

if main_container:
    accordion_items = main_container.find_all(
        "div",
        class_="accordion-item"
    )
else:
    print("Main container not found. Searching the entire page.")

    accordion_items = soup.find_all(
        "div",
        class_="accordion-item"
    )

print(f"Categories found: {len(accordion_items)}")

Categories found: 4


## 5. Extract the Winners

Iterate through the award categories, identify the winning entry, and extract the category, winner, and title.

In [9]:
from io import StringIO

# Extract HTML tables into DataFrames
tables = pd.read_html(StringIO(response.text))

print(f"Tables extracted: {len(tables)}")

# Select the winners table
df_winners = tables[0].copy()

# Keep only the required columns
df_winners = df_winners[
    ["Category", "Winner", "Title"]
].copy()

# Clean whitespace
for column in df_winners.columns:
    df_winners[column] = (
        df_winners[column]
        .astype("string")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Remove rows without a category
df_winners = df_winners.dropna(
    subset=["Category"]
).reset_index(drop=True)

print(f"Dataset shape: {df_winners.shape}")

display(df_winners.head(15))

Tables extracted: 1
Dataset shape: (85, 3)


,Category,Winner,Title
0,Record Of The Year,Billie Eilish,Bad Guy
1,Album Of The Year,Billie Eilish,"When We All Fall Asleep, Where Do We Go?"
2,Song Of The Year,"Billie Eilish, Finneas O'Connell",Bad Guy
3,Best New Artist,Billie Eilish,<NA>
4,Best Pop Solo Performance,Lizzo,Truth Hurts
5,Best Pop Duo/Group Performance,"Montero Lamar Hill, Billy Ray Cyrus",Old Town Road
6,Best Traditional Pop Vocal Album,Elvis Costello & The Imposters,Look Now
7,Best Pop Vocal Album,Billie Eilish,"When We All Fall Asleep, Where Do We Go?"
8,Best Dance Recording,The Chemical Brothers,Got To Keep On
9,Best Dance/Electronic Album,The Chemical Brothers,No Geography


In [10]:
!pip install ftfy

In [11]:
import pandas as pd
import ftfy  # pip install ftfy

# Corregir mojibake + limpiar espacios
for column in df_winners.columns:
    df_winners[column] = (
        df_winners[column]
        .astype("string")
        .apply(lambda x: ftfy.fix_text(x) if pd.notna(x) else x)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

# Quitar filas sin categoría
df_winners = df_winners.dropna(subset=["Category"]).reset_index(drop=True)

# Normalizar campos vacíos en Title
df_winners["Title"] = df_winners["Title"].replace("", pd.NA)

## 7. Export the Results

Save the extracted Grammy winners to a CSV file for further analysis.

In [12]:
from pathlib import Path

output_path = Path(
    "/home/sebastiancelorio.dev/Projects/data-engineering/spotify-grammys-etl/data/raw/grammys_2019.csv"
)

df_winners.to_csv(output_path, index=False)

print(f"CSV exported successfully: {output_path}")

CSV exported successfully: /home/sebastiancelorio.dev/Projects/data-engineering/spotify-grammys-etl/data/raw/grammys_2019.csv
